In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import numpy as np
import scipy.sparse as sp
import scipy.io as sio
import matplotlib.pyplot as plt

from pymv.models import LMGEC
from pymv.data import (
    load_dataset,
    list_datasets,
    load_acm,
    load_dblp, 
    load_imdb
)
from pymv.data.dataset import MultiViewDataset

from pymv.preprocessing import (
    normalize_features,
    preprocess_graph,
    get_propagated_features,
    preprocess_features,
)

from pymv.metrics import (
    clustering_accuracy,
    clustering_f1_score,
    nmi_score,
    ari_score,
)
from pymv.experiment import Experiment
from pymv.visualization import plot_loss, plot_clusters

In [4]:
print("Available datasets:", list_datasets())

Available datasets: ['acm', 'amazon_photos', 'bdgp', 'coco_all', 'coco_all_clip', 'coco_all_resnet_minilm', 'coco_all_vit_minilm', 'coco_all_vit_sbert', 'coco_cross', 'coco_cross_clip', 'coco_cross_resnet_minilm', 'coco_cross_vit_minilm', 'coco_cross_vit_sbert', 'dblp', 'esp_game', 'flickr30k', 'iaprtc12', 'imdb', 'nus_wide', 'pascal', 'pascal_clip', 'pascal_resnet_minilm', 'pascal_vit_minilm', 'pascal_vit_sbert', 'prokaryotic', 'wiki']


## Helper Functions 

In [5]:
## i'll use it to print the results when i'm not using expirements 

def evaluate(name, y_true, y_pred):
    acc = clustering_accuracy(y_true, y_pred)
    nmi = nmi_score(y_true, y_pred)
    ari = ari_score(y_true, y_pred)
    print(f"--- {name} ---")
    print(f"ACC: {acc:.4f} | NMI: {nmi:.4f} | ARI: {ari:.4f}\n")

In [ ]:
## Load the new multimodal .mat datasets (DAG/<name>/<name>_multimodal.mat)
## Keys: X_image (CLIP), X_text (Gemma), A (sparse graph adjacency), labels

def inspect_mat_dataset(mat_path):

    mat = sio.loadmat(mat_path)

    features = {
        "X_image": mat["X_image"],   # CLIP image features  (N, 768)
        "X_text":  mat["X_text"],    # Gemma text features  (N, 3072)
    }

    graphs = {
        "A": mat["A"],               # graph adjacency (N, N), sparse
    }

    labels = mat["labels"].reshape(-1)

    # =====================================================
    # PRINT INFO
    # =====================================================
    print(f"--- Inspecting: {mat_path} ---")

    for key, value in features.items():
        print(f"[Feature] {key}: shape={value.shape}, dtype={value.dtype}")

    for key, value in graphs.items():
        print(f"[Graph] {key}: shape={value.shape}, sparse={sp.issparse(value)}")

    print(
        f"[Labels] shape={labels.shape}, "
        f"classes={len(np.unique(labels))}"
    )

    return features, graphs, labels


In [ ]:
def build_dataset(
    name,
    features,
    graphs,
    labels,
    feature_keys,
    adj_matrices=None,
    beta_value=1.0,
    tf_idf=False,        # dense CLIP/Gemma embeddings -> plain L2, NOT TF-IDF
):

    if adj_matrices is None:
        adj_matrices = [None] * len(feature_keys)

    Hs = []
    As = []

    for x_key, A in zip(feature_keys, adj_matrices):

        # LOAD FEATURE
        X = features[x_key]

        if isinstance(A, str):
            A = graphs[A]

        # PROPAGATION
        H = get_propagated_features(
            features=X,
            adj=A,
            beta=beta_value,
            tf_idf=tf_idf,
            center=True,
            scale=False,
        )

        Hs.append(H)
        As.append(A)

    return MultiViewDataset(
        Hs,
        As,
        labels,
        name=name,
    )


## Benchmark on the new multimodal datasets

For each dataset we run **LMGEC once** (multimodal: image + text, both propagated
over the graph `A` with `beta=1.0`), repeated for **10 runs** to get mean +/- std.


In [ ]:
# New multimodal datasets, stored at DAG/<name>/<name>_multimodal.mat
DATASETS = [
    "Toys", "Movies", "GroceryS", "RedditS",   # smaller -> run first
    "Photo", "Arts", "Grocery", "Reddit",      # larger
]

def mat_path_for(name):
    return f"DAG/{name}/{name}_multimodal.mat"


# Collect results across the per-dataset cells below.
results = {}

def run_lmgec(name):
    """Run LMGEC once (image + text + graph) on one dataset, 10 runs."""
    print("=" * 70)
    features, graphs, labels = inspect_mat_dataset(mat_path_for(name))
    k = len(np.unique(labels))

    # ONE LMGEC model
    model = LMGEC(
        n_clusters=k,
        embedding_dim=k + 1,
        preprocess=False,     # build_dataset already preprocessed the views
        temperature=1.0,
    )
    model.name = "LMGEC"

    # Multimodal view: image + text, each propagated over the graph
    ds = build_dataset(
        name=f"{name} (image+text+graph)",
        features=features,
        graphs=graphs,
        labels=labels,
        feature_keys=["X_image", "X_text"],
        adj_matrices=["A", "A"],
        beta_value=1.0,
    )

    print(f"\nLaunching LMGEC on {name} (10 runs) ...")
    exp = Experiment(
        datasets=[ds],
        models=[model],
        metrics=[clustering_accuracy, nmi_score, ari_score, clustering_f1_score],
        runs=10,
        verbose=2,
    )
    results[name] = exp.run()
    return results[name]


### Toys

In [ ]:
run_lmgec("Toys")


### Movies

In [ ]:
run_lmgec("Movies")


### GroceryS

In [ ]:
run_lmgec("GroceryS")


### RedditS

In [ ]:
run_lmgec("RedditS")


### Photo

In [ ]:
run_lmgec("Photo")


### Arts

In [ ]:
run_lmgec("Arts")


### Grocery

In [ ]:
run_lmgec("Grocery")


### Reddit

In [ ]:
run_lmgec("Reddit")
